In [1]:
import sys
sys.path.append('..')

import importlib
import agents.narrator as na
importlib.reload(na)
from agents.narrator import run_narrator

print("narrator imported")

narrator imported


In [2]:
fake_state = {
    "prediction": "mel",
    "confidence": 0.62,
    "shap_result": {
        "status": "success",
        "interpretation": "Low feature attribution — decision distributed broadly across image"
    },
    "gradcam_result": {
        "status": "success",
        "attention_region": "center of the lesion",
        "interpretation": "Model attention tightly focused with high gradient signal strength. High-attention coverage: 8.3% of image area."
    },
    "critique": "Agreement: both methods identified the central lesion area. Confidence assessment: moderate confidence with some uncertainty. Next focus: examine border irregularity features.",
    "contradictions": ["SHAP shows broad attribution but Grad-CAM shows tight focus — methods partially disagree"]
}

print("testing Narrator LLM with fake state...\n")
output = run_narrator(fake_state)

print("\nexplanation:")
print(output['explanation'])
print("\nconfidence note:")
print(output['confidence_note'])

testing Narrator LLM with fake state...

  [Narrator node] writing plain English explanation...
  [Narrator node] done. explanation length=547 chars

explanation:
The model predicted a melanoma with a moderate confidence level of 62%, suggesting it's not entirely certain about the diagnosis. The visual evidence supports this prediction, as the model focused on the center of the lesion with high attention, which is a common feature of melanomas, but the broad feature attribution and some contradictions between methods suggest there may be uncertainty about the diagnosis. Given these mixed signals, Human review is recommended to further examine the border irregularity features and confirm the diagnosis.

confidence note:
Low confidence — contradictions detected, human review recommended


In [3]:
clean_state = {
    "prediction": "nv",
    "confidence": 0.99,
    "shap_result": {
        "status": "success",
        "interpretation": "High feature concentration — model focused on specific localised regions strongly"
    },
    "gradcam_result": {
        "status": "success",
        "attention_region": "center of the lesion",
        "interpretation": "Model attention tightly focused with high gradient signal strength. High-attention coverage: 9.1% of image area."
    },
    "critique": "Agreement: both methods clearly identify the central mole as the key feature. Confidence assessment: high confidence well supported by visual evidence. Next focus: no further investigation needed.",
    "contradictions": []
}

print("testing with clean high confidence state...\n")
clean_output = run_narrator(clean_state)

print("\nexplanation:")
print(clean_output['explanation'])
print("\nconfidence note:")
print(clean_output['confidence_note'])

testing with clean high confidence state...

  [Narrator node] writing plain English explanation...
  [Narrator node] done. explanation length=407 chars

explanation:
The model predicted that the skin lesion is a common benign mole, known as a melanocytic nevi, with a high confidence level of 99.0%. This prediction is supported by visual evidence, as the model focused on the center of the lesion where a distinct mole is visible, and the high gradient signal strength indicates a clear boundary between the mole and the surrounding skin. This prediction appears reliable.

confidence note:
High confidence — explainability methods agree, prediction is reliable


In [4]:
import torch
from torchvision import transforms
from PIL import Image
import glob

import agents.graph as gm
importlib.reload(gm)
from agents.graph import xai_agent

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

images = glob.glob("../data/HAM10000_images_part_1/*.jpg")
test_tensor = transform(Image.open(images[7]).convert("RGB")).unsqueeze(0)

state = {
    "image_path":      images[7],
    "image_tensor":    test_tensor,
    "prediction":      "nv",
    "confidence":      0.99,
    "pred_class_idx":  0,
    "shap_result":     {},
    "gradcam_result":  {},
    "critique":        "",
    "contradictions":  [],
    "next_action":     "",
    "explanation":     "",
    "confidence_note": "",
    "loop_count":      0
}

print("running full graph — Planner + SHAP + GradCAM + LLM Critic + LLM Narrator")
print("this is the first time the whole pipeline runs with real LLM calls\n")

result = xai_agent.invoke(state)

print("\n" + "="*50)
print("FINAL EXPLANATION:")
print("="*50)
print(result['explanation'])
print("\nCONFIDENCE NOTE:")
print(result['confidence_note'])
print("\nCRITIQUE:")
print(result['critique'])

d:\nn-xai-agent\xai-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


XAI agent graph compiled successfully
XAI agent graph compiled successfully
running full graph — Planner + SHAP + GradCAM + LLM Critic + LLM Narrator
this is the first time the whole pipeline runs with real LLM calls

  [Planner] confidence=0.99, loop=0
  [Planner router] high confidence, going gradcam only
  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=center of the lesion
  [Critic node] reviewing SHAP and Grad-CAM with LLM...
  [Critic node] done. contradictions=1, loop=0
    - SHAP analysis failed to run, preventing a comparison with Grad-CAM results
  [Critic router] contradiction found, looping back to planner
  [Planner] confidence=0.99, loop=1
  [Planner router] high confidence, going gradcam only
  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=center of the lesion
  [Critic node] reviewing SHAP and Grad-CAM with LLM...
  [Critic node] done. contradictions=1, loop=1
    - SHAP analysis fail

In [6]:
import base64
import matplotlib.pyplot as plt
from PIL import Image
import io

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

shap_img = Image.open(
    io.BytesIO(base64.b64decode(result['shap_result']['plot_b64']))
) if result['shap_result'].get('plot_b64') else None

gradcam_img = Image.open(
    io.BytesIO(base64.b64decode(result['gradcam_result']['plot_b64']))
) if result['gradcam_result'].get('plot_b64') else None

if shap_img:
    axes[0].imshow(shap_img)
    axes[0].set_title("SHAP — feature attribution")
    axes[0].axis("off")
else:
    axes[0].text(0.5, 0.5, "SHAP not available", ha='center')
    axes[0].axis("off")

if gradcam_img:
    axes[1].imshow(gradcam_img)
    axes[1].set_title("Grad-CAM — visual attention")
    axes[1].axis("off")
else:
    axes[1].text(0.5, 0.5, "Grad-CAM not available", ha='center')
    axes[1].axis("off")

plt.suptitle(
    f"XAI Agent — prediction: {result['prediction']} ({result['confidence']*100:.1f}%)",
    fontsize=13
)
plt.tight_layout()
display(fig)
plt.close()

print("\nfull pipeline working:")
print("CNN -> SHAP -> Grad-CAM -> LLM Critic -> LLM Narrator -> plain English explanation")

<Figure size 1600x500 with 2 Axes>


full pipeline working:
CNN -> SHAP -> Grad-CAM -> LLM Critic -> LLM Narrator -> plain English explanation


In [7]:
print("testing on 3 different images to compare explanations\n")

for i, img_path in enumerate(images[10:13]):
    tensor = transform(Image.open(img_path).convert("RGB")).unsqueeze(0)

    s = {
        "image_path":      img_path,
        "image_tensor":    tensor,
        "prediction":      "nv",
        "confidence":      0.99,
        "pred_class_idx":  0,
        "shap_result":     {},
        "gradcam_result":  {},
        "critique":        "",
        "contradictions":  [],
        "next_action":     "",
        "explanation":     "",
        "confidence_note": "",
        "loop_count":      0
    }

    r = xai_agent.invoke(s)

    print(f"Image {i+1}: {os.path.basename(img_path)}")
    print(f"Explanation: {r['explanation']}")
    print(f"Note: {r['confidence_note']}")
    print()

testing on 3 different images to compare explanations

  [Planner] confidence=0.99, loop=0
  [Planner router] high confidence, going gradcam only
  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=center of the lesion
  [Critic node] reviewing SHAP and Grad-CAM with LLM...
  [Critic node] done. contradictions=1, loop=0
    - SHAP analysis failed to run, preventing a comparison with Grad-CAM results
  [Critic router] contradiction found, looping back to planner
  [Planner] confidence=0.99, loop=1
  [Planner router] high confidence, going gradcam only
  [GradCAM node] running Grad-CAM analysis...
  [GradCAM node] done. status=success, region=center of the lesion
  [Critic node] reviewing SHAP and Grad-CAM with LLM...
  [Critic node] done. contradictions=1, loop=1
    - SHAP analysis failed to run, preventing a comparison with Grad-CAM results
  [Critic router] going to narrator
  [Narrator node] writing plain English explanation...
  [Narrator nod

NameError: name 'os' is not defined